# Nemotron · Unsloth · 0.85-Repro (From-Scratch LoRA)

**Goal**: reproduce dgxchen's 0.85 LB recipe starting from the raw base model (no pre-trained adapter), using **our own offline package infrastructure** and **our own training data**.

## What we borrow from the 0.85 notebook
- **Framework**: Unsloth `FastLanguageModel` + `get_peft_model`
- **LoRA**: `r=32, alpha=32, dropout=0.0`, target_modules = `[q,k,v,o,in,out,up,down]_proj + lm_head`
- **Sequence length**: `max_seq_length=8192` (no truncation of long CoT)
- **Training**: 2 epochs, per-device batch 1, grad_accum 32 (effective 32), LR `2e-4` linear, warmup 0
- **Optimizer**: adam β1=0.9, β2=0.95, ε=1e-8, weight_decay=0, max_grad_norm=1e9 (no clip)
- **Packing**: False
- **Chat template**: `enable_thinking=True`, CoT goes in `<think>...</think>`, answer in `\boxed{}`; `\boxed{}` stripped from CoT
- **Stratified batching by `type`** (one type per effective batch when possible)
- **Prompt suffix**: official metric suffix (same as competition eval)

## What's different (on purpose)
- **Data**: our own dataset (`/kaggle/input/prog-cot-training-data/sft_thinking.csv` etc.) — this is the only open variable; notebook accepts either a `generated_cot` column (dgxchen-style) or a `thinking` column (our style).
- **Offline packages**: **100% from our own `hastws/sft-offline-packages`** — zero third-party Kaggle dataset / utility-script dependencies.
- **Single stage only** — no stage 2, no boxed-loss-weight tricks (clean reproduction).

## Offline package manifest (self-contained)
Our `offline_packages/` now ships everything needed on Blackwell:
- Training stack: `unsloth-2026.4.4`, `unsloth_zoo-2026.4.6`, `trl-0.29.1`, `tyro-1.0.12`, `bitsandbytes-0.49.2`, `annotated_doc`, `nest_asyncio`, `psutil-7.2.2`, `hf_transfer`
- **Blackwell-capable `triton-3.6.0`** wheel (bundles its own sm_120 ptxas — no external patching)
- **`mamba_ssm-2.3.1+cu12torch2.10`** and **`causal_conv1d-1.6.1+cu12torch2.10`** wheels (pre-compiled for the Kaggle PyTorch 2.10 + CUDA 12 image)

`kernel-metadata.json` only lists: the competition, the base model, `hastws/sft-offline-packages`, and our training-data dataset. No `mayukh18/nemotron-packages`, no `ryanholbrook/nvidia-utility-script`, no `dennisfong/...`, no `rubyducklove/...`.


In [ ]:
# ============================================================
# Cell 1 · Offline package install (our repo, fully self-contained)
# ============================================================
# Kaggle stripped "+" from uploaded filenames, so we stored them with "_PLUS_"
# and restore "+" here by copying into /tmp/wheels before pip install.
import subprocess, sys, os, glob, shutil

# Diagnostic: show what actually got mounted.
print("=== /kaggle/input tree ===")
for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.count("/") - 2
    if depth <= 2:
        print("  " * depth + os.path.basename(root) + "/")
        if depth == 2:
            for f in files[:3]:
                print("  " * (depth + 1) + f)
            if len(files) > 3:
                print("  " * (depth + 1) + f"... (+{len(files)-3} more)")

# Find any directory containing wheels (robust to Kaggle mount-name drift).
wheel_hits = glob.glob("/kaggle/input/**/*.whl", recursive=True)
# We only want OUR offline_packages wheels, not random wheels mounted from
# competition / model / other datasets.
REQUIRED = {"unsloth-", "trl-", "mamba_ssm-", "causal_conv1d-", "triton-"}
candidate_dirs = {}
for w in wheel_hits:
    d = os.path.dirname(w)
    candidate_dirs.setdefault(d, []).append(os.path.basename(w))
offline_dir = None
for d, names in candidate_dirs.items():
    if sum(any(n.startswith(pref) for n in names) for pref in REQUIRED) >= 3:
        offline_dir = d
        break
if not offline_dir:
    raise RuntimeError(
        f"Could not locate offline wheels. Scanned {len(wheel_hits)} whl files; "
        f"candidate dirs: {list(candidate_dirs.keys())}"
    )
print(f"\nOffline dir: {offline_dir}  ({len(candidate_dirs[offline_dir])} wheels)")

# Stage to /tmp/wheels, restoring "+" in filenames
STAGE_DIR = "/tmp/wheels"
if os.path.exists(STAGE_DIR):
    shutil.rmtree(STAGE_DIR)
os.makedirs(STAGE_DIR, exist_ok=True)

for src in sorted(glob.glob(os.path.join(offline_dir, "*.whl"))):
    name = os.path.basename(src).replace("_PLUS_", "+")
    dst = os.path.join(STAGE_DIR, name)
    shutil.copy2(src, dst)

whls = sorted(glob.glob(os.path.join(STAGE_DIR, "*.whl")))
print(f"Installing {len(whls)} wheels from {STAGE_DIR} (staged from {offline_dir})")

# triton wheel needs --force-reinstall to override Kaggle's bundled triton
# (our triton-3.6.0 ships a Blackwell-capable ptxas)
FORCE_REINSTALL_PREFIXES = ("triton-",)

for whl in whls:
    name = os.path.basename(whl)
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-deps"]
    if any(name.startswith(p) for p in FORCE_REINSTALL_PREFIXES):
        cmd.append("--force-reinstall")
    cmd.append(whl)
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
    status = "OK" if r.returncode == 0 else "FAIL"
    print(f"  {status}: {name}")
    if r.returncode != 0:
        if any(tok in name.lower() for tok in ("unsloth", "trl", "mamba_ssm", "causal_conv1d", "triton-")):
            raise RuntimeError(f"Critical wheel install failed: {name}\n{r.stderr[-600:]}")

# Verify
print("\n=== Package versions ===")
for pkg in ["unsloth", "unsloth_zoo", "trl", "peft", "datasets", "transformers",
            "bitsandbytes", "triton", "mamba_ssm", "causal_conv1d"]:
    try:
        mod = __import__(pkg)
        print(f"  {pkg}={getattr(mod, '__version__', '?')}")
    except Exception as e:
        if pkg in ("unsloth", "trl", "triton"):
            raise RuntimeError(f"FATAL: {pkg} not importable: {e}")
        print(f"  WARNING: {pkg}: {e}")


In [ ]:
# ============================================================
# Cell 2 · Imports + Blackwell RMSNorm fallback
# ============================================================
# NOTE: We install triton 3.6.0 from our offline_packages in Cell 1,
# which bundles a Blackwell-capable ptxas. No external ptxas patching needed.
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTHONIOENCODING"] = "utf-8"

import sys, shutil, gc, zipfile, time, json, math, re, random
import pandas as pd
import torch
import torch.nn.functional as F
from collections import defaultdict
from datasets import Dataset as HFDataset
import kagglehub

print(f"torch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    mem = getattr(props, "total_memory", 0)
    print(f"GPU: {torch.cuda.get_device_name(0)}  mem={mem/1024**3:.1f} GB  cc={props.major}.{props.minor}")

import triton
print(f"triton: {triton.__version__}")


# ---- Pure-python rmsnorm fallback (guards against broken/absent triton kernel) ----
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast:
        x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None:
        out = out + bias.float()
    if z is not None:
        out = out * F.silu(z.float())
    return out.to(dtype)

for _mod_name, _mod in list(sys.modules.items()):
    if hasattr(_mod, "rmsnorm_fn"):
        _mod.rmsnorm_fn = _pure_rmsnorm_fn

print("Environment ready.")


## Configuration (hyperparameters borrowed from 0.85 LB recipe)

In [ ]:
# ============================================================
# Cell 3 · Hyperparameters (copied from dgxchen 0.85 LB)
# ============================================================
SEED = 42

# --- LoRA ---
LORA_RANK      = 32
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.0
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "in_proj", "out_proj", "up_proj", "down_proj",
    "lm_head",
]

# --- Training ---
# NOTE: huikang 0.85 uses max_length=8192 because his reasoning/*.txt CoT data
# averages ~5000 chars/sample. Our sft_thinking.csv averages only ~950 chars
# (~300 tokens) with max ~2400 chars (~800 tokens). Using 8192 here wastes
# ~10x compute on padding. 2048 gives plenty of headroom.
MAX_SEQ_LEN               = 2048
NUM_TRAIN_EPOCHS          = 1      # huikang 0.85 also uses 1 epoch
PER_DEVICE_BATCH_SIZE     = 1
GRADIENT_ACCUMULATION     = 32      # effective batch = 32
LEARNING_RATE             = 2e-4
LR_SCHEDULER              = "linear"
WARMUP_STEPS              = 0
ADAM_BETA1                = 0.9
ADAM_BETA2                = 0.95
ADAM_EPSILON              = 1e-8
WEIGHT_DECAY              = 0.0
MAX_GRAD_NORM             = 1e9     # effectively disables gradient clipping
PACKING                   = True    # short samples pack well into 2048 window
LOGGING_STEPS             = 10
DATALOADER_NUM_WORKERS    = 2

# --- Dataset (our own, mounted via kernel-metadata dataset_sources) ---
# Must expose columns: prompt / answer / type  +  one of:
#   - "generated_cot"   (dgxchen-style)
#   - "thinking"        (our sft_thinking.csv style)
DATASET_PATH_DEFAULT = "/kaggle/input/prog-cot-training-data/sft_thinking.csv"

# --- Official prompt suffix (MUST match eval metric script verbatim) ---
PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

# --- Paths ---
BASE_MODEL_NAME_FOR_CONFIG = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
OUTPUT_DIR   = "/kaggle/working/sft_output"
ADAPTER_DIR  = "/kaggle/working/sft_adapter"
SUBMIT_DIR   = "/kaggle/working/submission_adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(ADAPTER_DIR, exist_ok=True)
os.makedirs(SUBMIT_DIR, exist_ok=True)

print("LoRA: r={}, alpha={}, dropout={}".format(LORA_RANK, LORA_ALPHA, LORA_DROPOUT))
print("target_modules:", LORA_TARGET_MODULES)
print("max_seq_len={}, epochs={}, batch={}x{}, packing={}".format(
    MAX_SEQ_LEN, NUM_TRAIN_EPOCHS, PER_DEVICE_BATCH_SIZE, GRADIENT_ACCUMULATION, PACKING))
print("lr={} ({} warmup={}), betas=({},{}), eps={}, wd={}, max_grad_norm={}".format(
    LEARNING_RATE, LR_SCHEDULER, WARMUP_STEPS,
    ADAM_BETA1, ADAM_BETA2, ADAM_EPSILON, WEIGHT_DECAY, MAX_GRAD_NORM))
print("prompt_suffix:", repr(PROMPT_SUFFIX))


## Load base model + attach LoRA (Unsloth, from scratch)

In [ ]:
# ============================================================
# Cell 4 · Load the raw base model via Unsloth (no pre-trained adapter)
# ============================================================
from unsloth import FastLanguageModel

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
print(f"Base model path: {MODEL_PATH}")

t0 = time.time()
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=False,
    load_in_8bit=False,
    full_finetuning=False,
    trust_remote_code=True,
    unsloth_force_compile=False,
    attn_implementation="sdpa",
    dtype=torch.bfloat16,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Base model loaded in {time.time()-t0:.1f}s.  vocab={len(tokenizer)}")

# Force-disable fast path on Nemotron-H modules (broken cuda kernels on Blackwell)
for _name, _mod in sys.modules.items():
    if "modeling_nemotron_h" in _name and hasattr(_mod, "is_fast_path_available"):
        _mod.is_fast_path_available = False

# Attach a fresh LoRA via Unsloth
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
model.print_trainable_parameters()

_total = sum(p.numel() for p in model.parameters())
_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {_total/1e9:.2f}B  trainable: {_train/1e6:.1f}M ({_train/_total*100:.2f}%)")


## Build training data (our dataset, same format as 0.85 recipe)

Format:
```
<user> prompt + PROMPT_SUFFIX
<assistant><think>{cot without any \boxed{} inside}
</think>
\boxed{answer}
```
Chat template uses `enable_thinking=True`. Loss is computed over the full assistant text (no boxed-loss-weighting).

In [ ]:
# ============================================================
# Cell 5 · Load dataset, strip \boxed{} from CoT, build records
# ============================================================
DATASET_PATH = "/kaggle/input/datasets/hastws/prog-cot-training-data/sft_thinking.csv"
print(f"Dataset: {DATASET_PATH}")

df = pd.read_csv(DATASET_PATH)
print(f"rows: {len(df)}  columns: {list(df.columns)}")

# Figure out which column holds the CoT
if "generated_cot" in df.columns:
    COT_COL = "generated_cot"
elif "thinking" in df.columns:
    COT_COL = "thinking"
else:
    raise KeyError(f"Expected 'generated_cot' or 'thinking' column in {DATASET_PATH}")

# Must have: prompt, answer, type, <cot col>
for _col in ("prompt", "answer", "type", COT_COL):
    if _col not in df.columns:
        raise KeyError(f"Missing column {_col!r} in {DATASET_PATH}")

# Shuffle
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

records, record_types = [], []
dropped = 0
for _, row in df.iterrows():
    prompt = str(row["prompt"])
    answer = str(row["answer"])
    cot = str(row[COT_COL]) if row[COT_COL] is not None else ""
    if not cot or cot == "nan" or len(cot.strip()) < 5:
        dropped += 1
        continue
    cot_cleaned = re.sub(r"\\boxed\{[^}]*\}", "", cot).rstrip()
    user_content = prompt + PROMPT_SUFFIX
    assistant_content = cot_cleaned + f"\n</think>\n\\boxed{{{answer}}}"
    records.append({"messages": [
        {"role": "user",      "content": user_content},
        {"role": "assistant", "content": assistant_content},
    ]})
    record_types.append(str(row["type"]))

print(f"SFT records: {len(records)}   dropped (no CoT): {dropped}")
print("Type distribution:")
for _t, _c in sorted(pd.Series(record_types).value_counts().items()):
    print(f"  {_t:14s} {_c}")

dataset = HFDataset.from_list(records)


def formatting_prompts_func(example):
    texts = []
    for conversation in example["messages"]:
        try:
            text = tokenizer.apply_chat_template(
                conversation, tokenize=False,
                add_generation_prompt=False, enable_thinking=True,
            )
        except TypeError:
            text = tokenizer.apply_chat_template(
                conversation, tokenize=False, add_generation_prompt=False,
            )
        texts.append(text)
    return {"text": texts}


print("Materializing text column...")
dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
    num_proc=4,
    desc="chat template",
)
print("Sample record:")
print(dataset[0]["text"][:800])


## Train (single stage, stratified by `type`)

In [ ]:
# ============================================================
# Cell 6 · SFT training (stratified batching by `type`)
# ============================================================
from torch.utils.data import DataLoader, Sampler
from trl import SFTTrainer, SFTConfig


def build_stratified_index_order(labels, batch_size, seed):
    """Spread each `type` across effective batches (mimics dgxchen 0.85)."""
    by_label = defaultdict(list)
    for idx, label in enumerate(labels):
        by_label[label].append(idx)

    rng = random.Random(seed)
    for idx_list in by_label.values():
        rng.shuffle(idx_list)

    n_batches = max(1, math.ceil(len(labels) / batch_size))
    batches = [[] for _ in range(n_batches)]
    batch_order = list(range(n_batches))
    rng.shuffle(batch_order)

    assigned = 0
    for label in sorted(by_label.keys()):
        for idx in by_label[label]:
            batches[batch_order[assigned % n_batches]].append(idx)
            assigned += 1

    order = [idx for batch in batches for idx in batch]
    if len(order) != len(labels):
        raise ValueError("stratified order size mismatch")
    return order


class PrecomputedOrderSampler(Sampler):
    def __init__(self, order):
        self.order = list(order)
    def __iter__(self):
        return iter(self.order)
    def __len__(self):
        return len(self.order)


class StratifiedSFTTrainer(SFTTrainer):
    def __init__(self, *args, stratified_order=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.stratified_order = stratified_order

    def get_train_dataloader(self):
        if self.train_dataset is None:
            raise ValueError("train_dataset is None")
        if self.stratified_order is None:
            return super().get_train_dataloader()
        if len(self.stratified_order) != len(self.train_dataset):
            raise ValueError("stratified order length mismatch")
        kwargs = {
            "batch_size": self.args.per_device_train_batch_size,
            "sampler": PrecomputedOrderSampler(self.stratified_order),
            "collate_fn": self.data_collator,
            "num_workers": self.args.dataloader_num_workers,
            "pin_memory": self.args.dataloader_pin_memory,
            "persistent_workers": self.args.dataloader_persistent_workers,
            "drop_last": self.args.dataloader_drop_last,
        }
        if self.args.dataloader_num_workers > 0:
            kwargs["prefetch_factor"] = self.args.dataloader_prefetch_factor
        return DataLoader(self.train_dataset, **kwargs)


training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_steps=WARMUP_STEPS,
    max_length=MAX_SEQ_LEN,
    adam_beta1=ADAM_BETA1,
    adam_beta2=ADAM_BETA2,
    adam_epsilon=ADAM_EPSILON,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    logging_steps=LOGGING_STEPS,
    save_strategy="no",
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=DATALOADER_NUM_WORKERS,
    remove_unused_columns=False,
    seed=SEED,
    report_to="none",
    packing=PACKING,
)

effective_batch = max(1, PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION)
stratified_order = build_stratified_index_order(record_types, effective_batch, SEED)
print(f"Effective batch size: {effective_batch}")
print(f"Total steps ~= {math.ceil(len(dataset) / effective_batch) * NUM_TRAIN_EPOCHS}")

trainer = StratifiedSFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    formatting_func=formatting_prompts_func,
    stratified_order=stratified_order,
)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    print(f"GPU allocated before train: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

t0 = time.time()
trainer.train()
elapsed = time.time() - t0
print(f"Training done in {elapsed/60:.1f} min")

if torch.cuda.is_available():
    torch.cuda.synchronize()
    print(f"Peak GPU memory: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")

# Save adapter
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to {ADAPTER_DIR}")
for f in sorted(os.listdir(ADAPTER_DIR)):
    fp = os.path.join(ADAPTER_DIR, f)
    if os.path.isfile(fp):
        print(f"  {f:40s} {os.path.getsize(fp)/1024:.1f} KB")


## Package `submission.zip`

In [ ]:
# ============================================================
# Cell 7 · Build /kaggle/working/submission.zip
# ============================================================
required_files = ["adapter_config.json", "adapter_model.safetensors"]

for fname in required_files:
    src = os.path.join(ADAPTER_DIR, fname)
    dst = os.path.join(SUBMIT_DIR, fname)
    if not os.path.exists(src):
        raise FileNotFoundError(f"Missing adapter file: {src}")
    shutil.copy2(src, dst)
    print(f"Copied {fname}  ({os.path.getsize(dst)/1024/1024:.1f} MB)")

# Rewrite adapter_config.json — point base model to canonical HF id, force inference_mode + dropout=0
cfg_path = os.path.join(SUBMIT_DIR, "adapter_config.json")
with open(cfg_path) as f:
    cfg = json.load(f)

cfg["base_model_name_or_path"] = BASE_MODEL_NAME_FOR_CONFIG
cfg["inference_mode"] = True
cfg["lora_dropout"] = 0.0

with open(cfg_path, "w") as f:
    json.dump(cfg, f, indent=2)

print("\nFinal adapter_config.json:")
for k in ("r", "lora_alpha", "lora_dropout", "target_modules",
          "target_parameters", "base_model_name_or_path"):
    if k in cfg:
        val = cfg[k]
        if isinstance(val, list) and len(val) > 8:
            val = f"{val[:8]}... ({len(val)} items)"
        print(f"  {k}: {val}")

zip_path = "/kaggle/working/submission.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in required_files:
        zf.write(os.path.join(SUBMIT_DIR, fname), fname)

print(f"\nsubmission.zip: {os.path.getsize(zip_path)/1024/1024:.1f} MB  ->  {zip_path}")

with zipfile.ZipFile(zip_path) as zf:
    print("contents:", zf.namelist())
